<a href="https://colab.research.google.com/github/simichakravarty16/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  
**Name:** Simi Chakravarty
---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item of an order. In orders_json, each order contains one line which simultaneously represents one item. Once orders_json is joined to to vendors_json by vendor_id, this will remain the same.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [8]:
import pandas as pd

orders = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(orders) == 4
assert orders['qty'].sum() == 7

orders.head()

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.json_normalize(vendors_json)

joined = orders.merge(vendors, how='left', on='vendor_id', indicator=True)
joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined_df = joined.drop(columns = '_merge').copy()
joined_df['name'] = joined_df['name'].fillna('Unknown Name')
joined_df['zone'] = joined_df['zone'].fillna('Unknown Zone')
joined_df.groupby('name')['qty'].sum()

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unknown Name,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
joined_df.groupby('zone')['qty'].sum()

,qty
zone,
A,4
Unknown Zone,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [9]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

rows = []

for order in ragged_json:
    lines = order.get('lines', [])

    if lines:
        for line in lines:
            rows.append({
                'order': order['order'],
                'vendor_id': order['vendor_id'],
                'item': line.get('item'),
                'qty': line.get('qty')
            })
    else:
        rows.append({
            'order': order['order'],
            'vendor_id': order['vendor_id'],
            'item': None,
            'qty': None
        })

ragged = pd.DataFrame(rows)

ragged

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,5,V-10,None,NaN
2,6,V-01,Fries,NaN


### Q6 — Validate

In [10]:
assert len(orders) == 4
assert orders['qty'].sum() == 7
assert len(joined) == len(orders), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

_your answer here_